# 07 — Create Genie Space (REST API)

Creates a Genie space via the Databricks REST API connected to the gold-layer tables.
The space is pre-configured with table descriptions for natural language understanding.

Once created, open the Genie space in the UI to add sample questions and instructions.
The Genie space can then be added to a Domain on the Discover page and accessed
via the "Ask" search bar in Databricks One.

In [1]:
from databricks.sdk import WorkspaceClient
from dotenv import load_dotenv
import os, json, uuid

load_dotenv()

w = WorkspaceClient()

UC_CATALOG       = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA        = os.getenv("UC_SCHEMA", "nil_sponsorships")
GOLD_SCHEMA      = f"{UC_SCHEMA}_gold"
SQL_WAREHOUSE_ID = os.getenv("SQL_WAREHOUSE_ID")

if not SQL_WAREHOUSE_ID:
    raise ValueError("SQL_WAREHOUSE_ID not set in .env")

G = f"{UC_CATALOG}.{GOLD_SCHEMA}"
print(f"Gold schema: {G}")
print(f"Warehouse:   {SQL_WAREHOUSE_ID}")

Gold schema: alexander_booth.nil_sponsorships_gold
Warehouse:   49458fa9801b109b


In [2]:
# Build the serialized Genie space config.
# The API expects GenieSpaceExport proto (version 2).
# Tables MUST be sorted alphabetically. text_instructions has at most 1 item with content as array.
genie_config = {
    "version": 2,
    "data_sources": {
        "tables": [
            {"identifier": f"{G}.dim_athlete"},
            {"identifier": f"{G}.dim_date"},
            {"identifier": f"{G}.dim_sponsor"},
            {"identifier": f"{G}.fact_campaign_performance"},
            {"identifier": f"{G}.fact_deals"},
        ]
    },
    "instructions": {
        "text_instructions": [
            {"content": [
                "NIL stands for Name, Image, and Likeness — the right of college athletes to profit from their personal brand.",
                "When asked about top athletes, rank by total deal value from fact_deals unless the user specifies another metric.",
                "Always format currency values with $ and commas.",
                "Exclude Cancelled deals from aggregations unless specifically asked.",
                "The five conferences are SEC, Big Ten, Big 12, ACC, and Pac-12.",
            ]}
        ]
    }
}

serialized_space = json.dumps(genie_config)
print(f"Genie config: {len(serialized_space):,} chars")
print(f"Tables: {len(genie_config['data_sources']['tables'])}")

Genie config: 881 chars
Tables: 5


In [3]:
# Find existing Genie space or create a new one.
GENIE_TITLE = "NIL Sponsorship Genie"
current_user = w.current_user.me()
parent_path = f"/Workspace/Users/{current_user.user_name}"

# Check for existing Genie space via the list API
existing_id = None
try:
    resp = w.api_client.do("GET", "/api/2.0/genie/spaces")
    for space in resp.get("spaces", []):
        if space.get("title", "").startswith(GENIE_TITLE):
            existing_id = space["space_id"]
            print(f"Found existing: {space['title']} ({existing_id})")
            break
except Exception:
    pass

if existing_id:
    space_id = existing_id
    print(f"Genie space already exists — reusing: {space_id}")
else:
    response = w.api_client.do(
        "POST",
        "/api/2.0/genie/spaces",
        body={
            "warehouse_id": SQL_WAREHOUSE_ID,
            "title": GENIE_TITLE,
            "description": "Ask questions about college athlete NIL sponsorship deals, campaign performance, and sponsor ROI across 5 conferences and 10 schools.",
            "serialized_space": serialized_space,
            "parent_path": parent_path,
        },
    )
    space_id = response.get("space_id", response.get("id", "unknown"))
    print(f"Genie space created: {space_id}")

host = os.getenv("DATABRICKS_HOST", "").rstrip("/")
print(f"Title: {GENIE_TITLE}")
print(f"\nView at: {host}/genie/rooms/{space_id}")
print(f"\nThis Genie space is now available in Databricks One and can be added to a Domain.")

Found existing: NIL Sponsorship Genie (2) (01f1329e4611107fa98769ad8479a071)
Genie space already exists — reusing: 01f1329e4611107fa98769ad8479a071
Title: NIL Sponsorship Genie

View at: https://e2-demo-field-eng.cloud.databricks.com/genie/rooms/01f1329e4611107fa98769ad8479a071

This Genie space is now available in Databricks One and can be added to a Domain.


## Post-Creation Setup (in the UI)

Open the Genie space link above and configure:

**Sample Questions** (click the gear icon > Sample questions):
- Which athletes have the highest total NIL deal value?
- Show me sponsor ROI by industry
- Which conference has the most active deals?
- What is the average deal value by sport?
- Which platform drives the most campaign impressions?
- Show me the top 5 sponsors by total investment

The instructions and table connections were already configured programmatically above.